In [33]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict,Literal,Annotated 
from dotenv import load_dotenv
import os
load_dotenv()
import operator
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

In [34]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

model = ChatGoogleGenerativeAI(model="gemini-1.5-flash", google_api_key=GEMINI_API_KEY)
model2 = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model="gpt-4o")

In [35]:
generate_llm = ChatOpenAI(model="gpt-4o", openai_api_key=OPENAI_API_KEY)
evaluate_llm = ChatOpenAI(model="gpt-4o", openai_api_key=OPENAI_API_KEY)
optimizer_llm = ChatOpenAI(model="gpt-4o", openai_api_key=OPENAI_API_KEY)

In [36]:
from pydantic import BaseModel , Field

class TweetEvaluation (BaseModel):
    evaluation:Literal['approved','needs_improvement'] = Field('final evaluation of the tweet')
    feedback: str = Field(...,description='give me the feedback')

structured_evaluation_llm = evaluate_llm.with_structured_output(TweetEvaluation)

In [37]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation:Literal["approved","needs_improvement"]
    feedback:str
    iteration:int
    max_iterations:int

    tweet_history:Annotated[list[str],operator.add]
    feedback_history:Annotated[list[str],operator.add]

def generate_tweet(state:TweetState):

    message = [SystemMessage(content="u r funny and clever twitter influenser"),HumanMessage(content=f"""
write a short , original , and hilarious tweet on topic : "{state['topic']}" .

Rules : 
- do not use question-answer format 
- max 200 characters 
- use obseravtion humor 
- use simple day to day english 
""")]
    
    response = generate_llm.invoke(message).content

    return {'tweet':response , 'tweet_history':[response]}

def evalutaion_tweet(state:TweetState):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]
    response = structured_evaluation_llm.invoke(messages).content
    return {'evaluation':response.evaluation,'feedback':response.feedback,'feedback_history':[response.feedback]}



def optimizer_tweet(state:TweetState):
    
    messages = [
    SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
    HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]
    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration']+1
    return {'tweet':response,'iteration':iteration , 'tweet_history':[response]}




def route_evaluator(state:TweetState):
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iterations']:
        return END
    else:
        return "optimizer"

In [ ]:
graph = StateGraph(TweetState)

graph.add_node("generate",generate_tweet)
graph.add_node("evaluation",evalutaion_tweet)
graph.add_node("optimizer",optimizer_tweet)

graph.add_edge(START,"generate")
graph.add_edge("generate","evaluation")
graph.add_conditional_edges("evaluation",route_evaluator,{'approved':END , 'need_improvement':"optimizer"})
graph.add_edge("optimizer","evaluation")

workflow = graph.compile()
workflow

In [ ]:
initial_state = {
    'topic': 'indian railway',
    'tweet': '',
    'evaluation': '',
    'feedback': '',
    'iteration': 0,
    'max_iterations': 3
}

workflow.invoke(initial_state)